<div style="background: linear-gradient(135deg, #1a1a2e 0%, #2d0a16 100%); padding: 32px; border-radius: 12px; text-align: center; font-family: monospace;">
<h1 style="color: #FFD700; font-size: 2.6em; margin: 0; letter-spacing: 3px;">🌿 THE NEIGHBOR GAMES 🌿</h1>
<h2 style="color: #ffffff; font-size: 1.2em; margin: 10px 0 0 0; font-weight: normal; letter-spacing: 1px;">k-Nearest Neighbor Prediction Championship</h2>
<p style="color: #aaaacc; margin: 10px 0 0 0; font-size: 0.9em;">Elements of Data Science · Temple University</p>
<p style="color: #FFD700; margin: 8px 0 0 0; font-size: 0.85em; font-style: italic;">"May the best features be ever in your favor."</p>
</div>

---

### 🏆 Competition Overview

| Round | Task | What you do |
|---|---|---|
| 🌿 Tribute Round | Shared baseline — run as-is | Execute only |
| ⚔️ Arena Round | Tune `k` and features | **Fill in a function call** |
| 🎯 Prize Round | Predict bp of real molecules | **Build feature arrays + compute score** |
| 🌟 Wildcard | Inverse-distance weighting | Execute only |

After each round, click **Submit Score** to send your result to the instructor's live scorecard.


### 📛 Enter your team name

In [ ]:
team_name = "..."   # e.g. "District 4: The Alkanol Alliance"

---
### ⚙️ Setup — Run this first

In [ ]:
import numpy as np
from datascience import *
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('ggplot')
import ipywidgets as widgets
from IPython.display import display, HTML
import json, os, time, tempfile

Temple_color = '#9E1B34'
Gold_color   = '#FFD700'
print("✅ Modules loaded")

---
### 📡 Scorecard Connection
The path below must match the one in the instructor's scorecard notebook.  
Your instructor will tell you the correct path — leave it as-is unless told otherwise.


In [ ]:
# ── Set by instructor — must match scorecard notebook ────────────
SCORES_FILE = '/home/jovyan/shared-readwrite/neighbor_games/scores.json'
# ─────────────────────────────────────────────────────────────────

os.makedirs(os.path.dirname(SCORES_FILE), exist_ok=True)

def submit_score(round_name, score, extra=None):
    """Append this team's score for a round to the shared scores file."""
    if team_name.strip() in ('', '...'):
        print("⚠️  Set your team_name first!")
        return
    entry = {
        'team'      : team_name.strip(),
        'round'     : round_name,
        'score'     : round(float(score), 3),
        'timestamp' : time.strftime('%H:%M:%S'),
    }
    if extra:
        entry.update(extra)

    # Read existing scores (safe if file missing)
    try:
        with open(SCORES_FILE) as f:
            scores = json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        scores = []

    # Replace any prior submission from this team for this round
    scores = [s for s in scores if not (s['team'] == entry['team'] and s['round'] == round_name)]
    scores.append(entry)

    # Atomic write (temp file → rename) to avoid corruption during simultaneous saves
    dir_ = os.path.dirname(SCORES_FILE)
    with tempfile.NamedTemporaryFile('w', dir=dir_, delete=False, suffix='.tmp') as f:
        json.dump(scores, f, indent=2)
        tmp = f.name
    os.replace(tmp, SCORES_FILE)

def make_submit_button(round_name, score_var_name, extra_fn=None):
    """Return a styled submit button for a given round."""
    btn = widgets.Button(
        description=f'📡 Submit {round_name} score',
        button_style='',
        layout=widgets.Layout(width='260px', height='36px'),
        style={'button_color': '#9E1B34', 'font_weight': 'bold'}
    )
    out = widgets.Output()

    def on_click(_):
        score = globals()[score_var_name]
        extra = extra_fn() if extra_fn else None
        with out:
            out.clear_output()
            try:
                submit_score(round_name, score, extra)
                display(HTML(
                    f'<div style="color:#2E7D50;font-weight:bold;padding:6px 0;">'
                    f'✅ Submitted! Team <em>{team_name}</em> · {round_name} · {score:.3f} K</div>'
                ))
            except Exception as e:
                display(HTML(f'<div style="color:#9E1B34;">⚠️ Error: {e}</div>'))

    btn.on_click(on_click)
    display(widgets.VBox([btn, out]))

print("✅ Submit system ready — scores will go to:", SCORES_FILE)

---
### 🔧 Your Toolkit

In [ ]:
def distance(pt1, pt2):
    return np.sqrt(sum((pt1 - pt2) ** 2))

def row_distance(row1, row2):
    return distance(np.array(row1), np.array(row2))

def distances(training, test, target, features):
    dists = []
    for row in training.select(features).rows:
        dists.append(row_distance(row, test))
    return training.with_column('Distance', dists)

def closest(training, test, k, target, features):
    return distances(training, test, target, features).sort('Distance').take(np.arange(k))

def predict_knn(row, train, test, k=5, pr=False):
    if pr: print(f'row={row}, k={k}, features={features}')
    return np.average(
        closest(train, test.select(features).row(row), k, target, features).column(target[0])
    )

def predict_knn_weighted(row, train, test, k=5, pr=False):
    dist_table = closest(train, test.select(features).row(row), k, target, features)
    weights = 1 / (dist_table['Distance'] + 1e-9)
    return np.sum(dist_table[target[0]] * weights) / np.sum(weights)

print("✅ Tools loaded")

---
### 📂 Data & Competition Split

In [ ]:
ROH_data = Table().read_table('data/ROH_data.csv')
np.random.seed(42)
shuffled  = ROH_data.sample(with_replacement=False)
split_n   = int(0.75 * shuffled.num_rows)
train_raw = shuffled.take(np.arange(split_n))
test_raw  = shuffled.take(np.arange(split_n, shuffled.num_rows))
print(f"Training: {train_raw.num_rows} rows  |  Test: {test_raw.num_rows} rows")
print("Features:", [c for c in ROH_data.labels if c != 'bp'])

---
### 📏 Standardization

$$z = \frac{x - \mu_{\text{train}}}{\sigma_{\text{train}}}$$

> ⚠️ Compute $\mu$ and $\sigma$ from **training data only**, then apply to both sets.


In [ ]:
all_features = ["MW", "degree", "carbons"]
print(f"{'Feature':<12} {'Min':>8} {'Max':>8} {'Mean':>8} {'Std':>8}")
print("─" * 50)
for f in all_features:
    col = train_raw.column(f)
    print(f"{f:<12} {np.min(col):>8.2f} {np.max(col):>8.2f} {np.mean(col):>8.2f} {np.std(col):>8.2f}")

<div style="background:#fff8e1; border-left:4px solid #FFD700; padding:12px 16px; border-radius:0 8px 8px 0;">
<strong>🖊️ Challenge 1 of 4 — Apply the z-score formula</strong><br>
Fill in the standardization formula for both the training and test columns.<br>
Recall: &nbsp; $z = (x - \mu) \;/\; \sigma$
</div>


In [ ]:
train_stats = {}
for f in all_features:
    col = train_raw.column(f)
    train_stats[f] = (np.mean(col), np.std(col))

train = train_raw.select('bp')
test  = test_raw.select('bp')

for f in all_features:
    mu, sigma = train_stats[f]
    # 🖊️  Fill in the z-score formula for train and test:
    train = train.with_columns(f, ...)
    test  = test.with_columns( f, ...)

# Verify
print(f"{'Feature':<12} {'Mean':>10} {'Std':>10}   (should be ≈0, ≈1)")
print("─" * 48)
for f in all_features:
    print(f"{f:<12} {np.mean(train.column(f)):>10.4f} {np.std(train.column(f)):>10.4f}")

In [ ]:
def compute_score(predictions, label='Score'):
    actual = test.column('bp')
    preds  = np.array(predictions)
    if len(preds) != len(actual):
        print(f"⚠️  Need {len(actual)} predictions, got {len(preds)}")
        return None
    rmse = np.sqrt(np.mean((actual - preds) ** 2))
    fill = int(max(0, 28 - rmse / 2))
    print(f"{'─'*52}\n  {label}\n  RMSE = {rmse:.3f} K   [{'█'*fill}{'░'*(28-fill)}]\n{'─'*52}")
    return round(rmse, 3)

---
## 🌿 Tribute Round — Run exactly as written

In [ ]:
features = ["MW"]
target   = ["bp"]
k        = 5

tribute_predictions = [predict_knn(i, train, test, k=k) for i in np.arange(test.num_rows)]
tribute_rmse = compute_score(tribute_predictions, label="Tribute Baseline (MW only, k=5)")

In [ ]:
actual_bp = test_raw.column('bp')
plt.figure(figsize=(6,5))
plt.scatter(actual_bp, tribute_predictions, color=Temple_color, alpha=0.7, edgecolors='white', s=60)
diag = np.linspace(min(actual_bp), max(actual_bp), 100)
plt.plot(diag, diag, '--', color=Gold_color, lw=1.5, label='Perfect')
plt.xlabel('Actual bp (K)'); plt.ylabel('Predicted bp (K)')
plt.title(f'Tribute  |  RMSE = {tribute_rmse:.2f} K')
plt.legend(); plt.tight_layout(); plt.show()

#### 📡 Submit your Tribute score

In [ ]:
make_submit_button('Tribute', 'tribute_rmse')

---
## ⚔️ Arena Round — Tune Your Model!

**Available features:** `"MW"`, `"degree"`, `"carbons"` (all pre-standardized)

<div style="background:#fff8e1; border-left:4px solid #FFD700; padding:12px 16px; border-radius:0 8px 8px 0; margin-top:12px">
<strong>🖊️ Challenge 2 of 4 — Fill in the predict_knn call</strong><br>
Complete the list comprehension. Signature: &nbsp;<code>predict_knn(row, train, test, k)</code>
</div>


In [ ]:
# ── STEP 1: choose your settings ──────────────────────────────────
features = ["MW", "degree"]    # try adding "carbons"
k        = 5                   # try 3, 7, 10, 15 …
target   = ["bp"]

# ── STEP 2: fill in the predict_knn call ──────────────────────────
arena_predictions = [predict_knn(..., ..., ..., k=k) for i in np.arange(test.num_rows)]

arena_rmse = compute_score(arena_predictions, label=f"Arena: features={features}, k={k}")

In [ ]:
actual_bp = test_raw.column('bp')
plt.figure(figsize=(6,5))
plt.scatter(actual_bp, arena_predictions, color='steelblue', alpha=0.7, edgecolors='white', s=60)
diag = np.linspace(min(actual_bp), max(actual_bp), 100)
plt.plot(diag, diag, '--', color=Gold_color, lw=1.5, label='Perfect')
plt.xlabel('Actual bp (K)'); plt.ylabel('Predicted bp (K)')
plt.title(f'Arena  |  k={k}, features={features}\nRMSE = {arena_rmse:.2f} K')
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# k-sweep
features_sweep = ["MW", "degree"]
k_values, rmse_list = np.arange(1,21), []
for kk in k_values:
    preds = [predict_knn(i, train, test, k=kk) for i in np.arange(test.num_rows)]
    rmse_list.append(np.sqrt(np.mean((test.column('bp') - np.array(preds))**2)))
best_k = k_values[np.argmin(rmse_list)]
plt.figure(figsize=(7,4))
plt.plot(k_values, rmse_list, 'o-', color=Temple_color, lw=2, ms=6)
plt.axvline(best_k, color=Gold_color, ls='--', label=f'Best k={best_k}')
plt.xlabel('k'); plt.ylabel('Test RMSE (K)')
plt.title(f'k vs RMSE  |  features={features_sweep}')
plt.legend(); plt.tight_layout(); plt.show()
print(f"Best k={best_k}  RMSE={min(rmse_list):.3f} K")

#### 📡 Submit your Arena score

In [ ]:
make_submit_button('Arena', 'arena_rmse',
    extra_fn=lambda: {'features': features, 'k': int(k)})

---
## 🏆 Prize Round — Predict Real Molecules!

| Molecule | MW (g/mol) | Degree | Carbons | Actual bp |
|---|---|---|---|---|
| **Ethanol** | 46.07 | 1 | 2 | 351.4 K |
| **Isopropanol (IPA)** | 60.10 | 2 | 3 | 355.4 K |

> ⚠️ Standardize using `train_stats` — apply the **training** mean and std to each raw value.


In [ ]:
features = ["MW", "degree"]   # ← your best features
k        = 5
target   = ["bp"]

<div style="background:#fff8e1; border-left:4px solid #FFD700; padding:12px 16px; border-radius:0 8px 8px 0;">
<strong>🖊️ Challenge 3 of 4 — Standardize Isopropanol</strong><br>
Ethanol is done as a worked example. Build <code>ipa_std</code> for isopropanol using the same pattern.<br>
Raw values: MW = 60.10, degree = 2, carbons = 3. Use <code>train_stats</code> and match the order of <code>features</code>.
</div>


In [ ]:
# Ethanol — worked example
ethanol_raw = {"MW": 46.07, "degree": 1, "carbons": 2}
ethanol_std = np.array([(ethanol_raw[f] - train_stats[f][0]) / train_stats[f][1] for f in features])
print("Ethanol standardized:", dict(zip(features, ethanol_std.round(4))))

# Isopropanol — your turn
ipa_raw = {"MW": 60.10, "degree": 2, "carbons": 3}
# 🖊️  Build ipa_std using the same pattern as ethanol_std:
ipa_std = ...

print("IPA standardized:    ", dict(zip(features, ipa_std.round(4))))

In [ ]:
# Ethanol
ethanol_neighbors = closest(train, ethanol_std, k, target, features)
ethanol_neighbors.show()
ethanol_bp_pred = np.average(ethanol_neighbors.column(target[0]))
ETHANOL_ACTUAL  = 351.4
print(f"Ethanol   predicted={ethanol_bp_pred:.1f} K  actual={ETHANOL_ACTUAL} K  error={abs(ethanol_bp_pred-ETHANOL_ACTUAL):.1f} K")

# IPA
ipa_neighbors = closest(train, ipa_std, k, target, features)
ipa_neighbors.show()
ipa_bp_pred = np.average(ipa_neighbors.column(target[0]))
IPA_ACTUAL  = 355.4
print(f"IPA       predicted={ipa_bp_pred:.1f} K  actual={IPA_ACTUAL} K  error={abs(ipa_bp_pred-IPA_ACTUAL):.1f} K")

<div style="background:#fff8e1; border-left:4px solid #FFD700; padding:12px 16px; border-radius:0 8px 8px 0;">
<strong>🖊️ Challenge 4 of 4 — Compute the Prize Score</strong><br>
Prize score = average absolute error across both molecules. Fill in the denominator.
</div>


In [ ]:
ethanol_error = abs(ethanol_bp_pred - ETHANOL_ACTUAL)
ipa_error     = abs(ipa_bp_pred     - IPA_ACTUAL)

# 🖊️  Average the two errors:
prize_score = (ethanol_error + ipa_error) / ...

print("="*52)
print(f"  🏆  PRIZE ROUND — {team_name}")
print("="*52)
print(f"  Ethanol error     : {ethanol_error:.2f} K")
print(f"  Isopropanol error : {ipa_error:.2f} K")
print(f"  Prize Score (avg) : {prize_score:.2f} K  ← lower wins!")
print("="*52)

fig, ax = plt.subplots(figsize=(6,4))
x, w = np.arange(2), 0.3
ax.bar(x-w/2, [ETHANOL_ACTUAL, IPA_ACTUAL], w, label='Actual',    color=Gold_color,   edgecolor='black', lw=0.5)
ax.bar(x+w/2, [ethanol_bp_pred, ipa_bp_pred], w, label='Predicted', color=Temple_color, edgecolor='black', lw=0.5, alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(['Ethanol\n(351.4 K)', 'IPA\n(355.4 K)'], fontsize=11)
ax.set_ylabel('Boiling Point (K)'); ax.set_title(f'Prize Round: {team_name}')
ax.legend(); plt.tight_layout(); plt.show()

#### 📡 Submit your Prize score

In [ ]:
make_submit_button('Prize', 'prize_score')

---
## 🌟 Wildcard Round — Inverse-Distance Weighting

In [ ]:
features = ["MW", "degree"]
k        = 5
target   = ["bp"]

wildcard_predictions = [predict_knn_weighted(i, train, test, k=k) for i in np.arange(test.num_rows)]
wildcard_rmse = compute_score(wildcard_predictions, label=f"Wildcard (weighted): features={features}, k={k}")

#### 📡 Submit your Wildcard score

In [ ]:
make_submit_button('Wildcard', 'wildcard_rmse')

---
## 💬 Reflection

1. **Why do we apply the training mean and std to the test set rather than recomputing them?**
2. **What does `i` represent in the list comprehension in Challenge 2?**
3. **Why can't we use `standard_units()` to standardize a single new molecule?**
4. **Did adding more features improve your Arena RMSE after standardization?**


*Your answers here:*

1. 

2. 

3. 

4. 